In [ ]:
import importlib
import torch
import numpy as np
import sys
import os
import matplotlib.pyplot as plt

model_name = "10D_Fernandes_Phelan"
relative_path = os.path.join('..', '..', 'dptorch')

notebook_dir = os.getcwd()
absolute_path = os.path.abspath(os.path.join(notebook_dir, relative_path))

sys.path.insert(0, absolute_path)

def _iter_num(p):
    try:
        return int(p.stem.split("_")[-1])
    except ValueError:
        return -1
import pathlib
list_all_Iter=list(pathlib.Path(os.path.abspath(f"data/10D")).rglob("*.pth"))
all_checkpoint_iters = [_iter_num(p) for p in list_all_Iter]

latest_checkpoint_num = max(all_checkpoint_iters)

#### what to load
saved_checkpoint_file = 4319
checkpoint_file = latest_checkpoint_num

model = importlib.import_module(f"{model_name}.Model")

# RNG
torch.manual_seed(123)


# load the specific checkpoint
m = model.SpecifiedModel.load(
    path=os.path.abspath(f"data/10D/Iter_{checkpoint_file}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name},
)
# load the specific checkpoint
m_prev = model.SpecifiedModel.load(
    path=os.path.abspath(f"data/10D/Iter_{checkpoint_file-1}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name},
)

sigma = m.cfg["model"]["params"]["sigma"]
beta = m.cfg["model"]["params"]["beta"]
n_types = m.cfg["model"]["params"]["n_types"]
gp_offset = m.cfg["model"]["params"]["GP_offset"]

### Error

In [ ]:
m_l2 = float(np.mean((m.metrics[list(m.metrics.keys())[-1]]["l2"]).numpy()))
m_inf = float(np.mean((m.metrics[list(m.metrics.keys())[-1]]["l_inf"]).numpy()))

table_path = os.path.join(notebook_dir, "10D_FP_pointwise_error_table.tex")
with open(table_path, "w", encoding="utf-8") as f:
    f.write(
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\begin{tabular}{lcc}\n"
        "\\hline\n"
        "10D Model & $L_2$ & $L_\\infty$ \\\\\n"
        "\\hline\n"
        f"Baseline & ${m_l2:.4e}$ & ${m_inf:.4e}$ \\\\\n"
        "\\hline\n"
        "\\end{tabular}\n"
        "\\end{table}\n"
    )

print(f"Saved LaTeX table to: {table_path}")

In [ ]:
error_data = np.loadtxt(f"data/10D/V_func_error_{checkpoint_file}.txt")
# note that sampling on the 10D state space is difficult, thus we take the difference at the sample points as a proxy for the error. the sampled error is too low to be informative.
L2 = np.mean((m.metrics[f"{checkpoint_file}"]['l2']).numpy()) 
Linf = np.mean((m.metrics[f"{checkpoint_file}"]['l_inf']).numpy())

sim_data = np.loadtxt(f"data/10D/simulation_{checkpoint_file}.txt")

sim_diff = sim_data[:, 14]


sim_L2 = np.sqrt(np.mean(sim_diff**2))
sim_Linf = np.max(sim_diff)


def format_latex_sci(value):
    mantissa, exponent = f"{value:.1e}".split("e")
    return rf"${float(mantissa):.1f}\cdot 10^{{{int(exponent)}}}$"


latex_table = f"""

\\begin{{tabular}}{{l|l|c|c}}
    \\hline \\hline
     \\text{{Error type}} & \\text{{$L_2$}} & \\text{{$L_\\infty$}} \\\\
    \\hline \\hline
    Criterion 2 (global error) & {{{format_latex_sci(L2)}}} & {{{format_latex_sci(Linf)}}} \\\\
    Criterion 3 (error along a simulated path) & {{{format_latex_sci(sim_L2)}}} & {{{format_latex_sci(sim_Linf)}}} \\\\
    \\hline

\\end{{tabular}}

""".strip()


print(latex_table)

with open("Table_4_error_table_10D.tex", "w", encoding="utf-8") as table_file:
    table_file.write(latex_table)

# latex_table

### Maximal value table

In [ ]:
max_vec = np.zeros(n_types)
for indxd in range(n_types):

    d = indxd
    mask = m.state_sample[:, -1] == d * torch.tensor(1.)
    max_vec[indxd] = torch.max(m.V_sample[mask] + gp_offset)
    sample = m.state_sample_all[mask,:]



# Format max_vec as LaTeX table
theta_cols = " & ".join([str(i+1) for i in range(n_types)])
max_vals = " & ".join([f"{val:.2f}" for val in max_vec])

max_value_table = f"""    \\begin{{tabular}}{{l|{'c|' * (n_types-1)}c}} 
    \hline \hline 
       $\\theta$ &  {theta_cols} \\\\
      \hline \hline
     $\max\\bar K$ & {max_vals}\\\\
     \hline
    \end{{tabular}}"""

print(max_value_table)

with open("Table_5_max_value_table_10D.tex", "w", encoding="utf-8") as table_file:
    table_file.write(max_value_table)

### BAL

In [ ]:
def bayesian_opt_criterion(m, eval_pt,discrete_state, target_p, rho, beta):

    # #compute bayesian optimization criteria
    mean_v = m.M[discrete_state][target_p].predict_mean(
                    eval_pt
                )
    var_v = m.M[discrete_state][target_p].predict_var(
                    eval_pt
                )

    #Deisenroth criterion
    out_vec = (rho * (mean_v) + beta / 2.0 * torch.log(var_v + 1e-15))

    return out_vec

In [ ]:
bal_util = 0
for indxt in range(n_types):
    mask_0 = m.state_sample[:,-1] == indxt
    no_init_samples = m.cfg["no_samples"]
    n_pts = m.state_sample[mask_0,:].shape[0]
    beta =1
    rho=1

    indxp = n_pts-no_init_samples - 1

    train_sample = m.state_sample[mask_0,:][:no_init_samples+indxp,:-1]
    train_v = m.V_sample[mask_0][:no_init_samples+indxp]
    m.M[indxt][0].set_train_data(
        train_sample,
        train_v,
        strict=False,
    )

    bal_util += bayesian_opt_criterion(m,m.state_sample[mask_0,:][no_init_samples+indxp:no_init_samples+indxp+1,:-1],indxt,0,rho,beta) / n_types

In [ ]:
bal_util

In [ ]:
n_pts